# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Loading the starter dataset (`data/raw/content_refresh_anonymized.csv`, Lane 2's smaller-but-complete slice — every column this baseline needs is already in it, so no warehouse round trip needed for a rule this simple). `is_declining_label` is rebuilt here **only as an evaluation label** for the signal checks and precision@K below — it is never a feature, and `trend_direction` / `trend_pct` never enter the score.

In [9]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/pretom26/ml_internship_flyrankAI.git"
REPO_DIR = Path("/content/ml_internship_flyrankAI")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "skills/README.md").exists():
            os.chdir(candidate)
            break

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Evaluation label ONLY — built the same honest way as the session notebook, never fed into the rule.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"{df.shape[0]:,} pages | overall decline rate: {df['is_declining_label'].mean():.3f}")

30,000 pages | overall decline rate: 0.542


## 1. My rule and its reason codes

**Plan.** Two signals get checked against the evaluation label before anything is coded. Both are the signal *behind* a real FlyRank flag from the session — I'm not inventing new ones:

1. **Staleness** — the signal behind the refresh flags. Claim: *pages that haven't been updated in a while decline more often.*
2. **CTR vs. position** — the signal behind the CTR-fix logic. Claim: *pages clicking worse than peers at the same search position decline more often.*

Each gets one bucket table with n, and a one-word verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.

### Signal check 1 — Staleness (behind the refresh flags)

`days_since_last_update` is heavy-tailed with a hard step in the data around 90 days (median is 20, but a large block of pages sits at 90+). Bucketing at that natural step — rather than the 180-day number I first guessed — is what the data actually supports.

In [10]:
df["staleness_bucket"] = np.where(
    df["days_since_last_update"] >= 90, "stale_90d_plus", "updated_lt_90d"
)

staleness_table = df.groupby("staleness_bucket")["is_declining_label"].agg(n="count", decline_rate="mean")
staleness_table

,n,decline_rate
staleness_bucket,,
stale_90d_plus,9345,0.608454
updated_lt_90d,20655,0.512031


**Verdict: CONFIRMED.** Stale pages (≥90 days since update, n=9,345) decline at **60.8%**, vs. **51.2%** for recently updated pages (n=20,655) — a real gap on two solidly-sized buckets, well above the ~50-row floor. Worth noting honestly: I first tried the 180-day threshold used in the session example, and at that cutoff the signal actually flips (stale n=174 declines *less*, 47.1% vs 54.2%) — a tiny, likely-noisy bucket. The 90-day split is both the larger sample and the one the data's own step supports, so that's the threshold the rule below uses.

### Signal check 2 — CTR vs. position (behind the CTR-fix logic)

A flat CTR threshold isn't fair across positions — position 3 and position 30 have very different "normal" CTRs. So "underperforming" is defined *relative to peers in the same `position_tier`*: below that tier's median CTR. Rows with no position data are excluded (they can't be compared).

In [11]:
valid = df[df["position_tier"] != "no_data"].copy()
expected_ctr_by_tier = valid.groupby("position_tier")["ctr"].transform("median")
valid["ctr_bucket"] = np.where(
    valid["ctr"] < expected_ctr_by_tier, "underperform_ctr", "at_or_above_tier_median"
)

ctr_table = valid.groupby("ctr_bucket")["is_declining_label"].agg(n="count", decline_rate="mean")
ctr_table

,n,decline_rate
ctr_bucket,,
at_or_above_tier_median,16921,0.502630
underperform_ctr,13079,0.593088


**Verdict: CONFIRMED.** Pages clicking below their position tier's median CTR (n=10,307) decline at **65.9%**, vs. **54.3%** for pages at or above their tier's median (n=11,699). Both buckets are large; the gap (≈12 points) is the clearest of the two checks.

### The rule, in plain words

> A page is worth reviewing first if it's **stale** (no update in 90+ days), still **visible** (≥500 impressions in 90 days — so there's an audience to lose), and its **click-through rate is underperforming** its own position tier. Rank the pages that clear all three bars by how much exposure (impressions) is riding on them.

Both signal checks above CONFIRMED, so both go into the rule — staleness says *this page is neglected*, CTR-vs-position says *something on the page or snippet is actively costing clicks it should be getting*. Together they point at the same page for two independent reasons, which is a stronger flag than either alone.

**Reason code (one, fixed):** `stale_visible_ctr_underperform` — every flagged row scores for the same reason, by design; that's what makes a hand rule readable in one sentence. Rows that don't clear all three bars get `no_flag`.

**Action label:** `refresh_review` for flagged rows, `monitor` otherwise.

## 2. Build the ranked queue (writes the CSV)

In [12]:
stale = (df["days_since_last_update"] >= 90).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

pos_median_ctr = df[df["position_tier"] != "no_data"].groupby("position_tier")["ctr"].median()
expected_ctr = df["position_tier"].map(pos_median_ctr)
ctr_underperform = ((df["position_tier"] != "no_data") & (df["ctr"] < expected_ctr)).astype(int)

# Readable on purpose: score is 0 unless all three bars clear, then it's ranked by exposure.
df["action_score"] = stale * visible * ctr_underperform * df["impressions_90d"]
df["reason_code"] = np.where(df["action_score"] > 0, "stale_visible_ctr_underperform", "no_flag")
df["action_label"] = np.where(df["action_score"] > 0, "refresh_review", "monitor")

print(f"pages flagged for review: {(df['action_score'] > 0).sum():,} of {len(df):,}")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining_label"].values
base_rate = y.mean()
print(f"base rate (declining, unranked): {base_rate:.3f}")
for k in (10, 20, 50, 100):
    print(f"Precision@{k}: {precision_at_k(df['action_score'], y, k):.3f}")

pages flagged for review: 2,019 of 30,000
base rate (declining, unranked): 0.542
Precision@10: 0.700
Precision@20: 0.650
Precision@50: 0.520
Precision@100: 0.530


In [13]:
ranked = df.sort_values("action_score", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)

out_cols = ["rank", "content_id", "client_id", "action_score", "reason_code", "action_label",
            "impressions_90d", "days_since_last_update", "avg_position", "position_tier", "ctr",
            "trend_direction"]

out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
ranked[out_cols].to_csv(out_path, index=False)
print(f"wrote {len(ranked):,} rows to {out_path}")

wrote 30,000 rows to work/outputs/baseline_action_score.csv


**Reading the precision@K.** At K=10 and K=20 the rule clearly beats the 54.2% base rate — the top of the list is where the two CONFIRMED signals earn their keep. By K=50/100 it's basically back at the base rate: only ~2,000 pages clear all three bars, so past that the ranking runs out of true signal and starts padding with weaker exposure ties. That's the honest shape a hand rule is supposed to have, and it's exactly what Week 5's model needs to beat — not just at the very top, but deeper into the list.

## 3. Top-10 review

In [14]:
review_cols = ["rank", "content_id", "client_id", "impressions_90d", "days_since_last_update",
               "avg_position", "position_tier", "ctr", "trend_direction", "action_label"]
ranked[review_cols].head(10)

,rank,content_id,client_id,impressions_90d,days_since_last_update,avg_position,position_tier,ctr,trend_direction,action_label
0,1,content_5fe46e04994d,client_4e07408562,517715,104,4.2,page_1,0.14,down,refresh_review
1,2,content_36ff89c8214e,client_19581e27de,295097,104,7.3,page_1,0.05,stable,refresh_review
2,3,content_c8e9d6ab9013,client_19581e27de,208678,104,9.7,page_1,0.00,down,refresh_review
3,4,content_a7427266c305,client_19581e27de,201111,104,5.7,page_1,0.11,stable,refresh_review
4,5,content_91652435f57a,client_19581e27de,159590,104,7.8,page_1,0.06,stable,refresh_review
5,6,content_f42eb861c6dd,client_19581e27de,152467,104,6.5,page_1,0.13,down,refresh_review
6,7,content_11fcfd65d94c,client_19581e27de,149083,104,6.2,page_1,0.15,down,refresh_review
7,8,content_97a86caf3a3d,client_19581e27de,147670,104,6.4,page_1,0.07,down,refresh_review
8,9,content_8b36799b7e44,client_6208ef0f77,141400,104,32.0,page_3_5,0.02,down,refresh_review
9,10,content_c1fe78bc4e37,client_19581e27de,134055,104,7.5,page_1,0.03,down,refresh_review


1. **#1** `content_5fe46e04994d` — `refresh_review`: page_1 position (4.2), 517,715 impressions, CTR 0.14 vs a 0.16 tier median, stale at 104 days; label agrees (`down`). Wrong if the CTR gap is a snippet/SERP-feature artifact (e.g. a competing featured snippet) rather than anything on-page.
2. **#2** `content_36ff89c8214e` — `refresh_review`: page_1, CTR 0.05 vs 0.16 median, huge exposure (295k impressions); trend is `stable`, not `down` — a genuine miss for the label, though a 3x CTR gap is still worth a look. Wrong if this client's page-1 CTRs are structurally lower for a reason unrelated to page quality (e.g. branded query cannibalizing clicks elsewhere).
3. **#3** `content_c8e9d6ab9013` — `refresh_review`: CTR is literally 0.00 at page_1 position 9.7; label agrees (`down`). Wrong if `ctr = 0` here means "no clicks logged yet" rather than "zero clicks despite impressions" — worth a raw-count sanity check before treating it as a strong signal.
4. **#4** `content_a7427266c305` — `refresh_review`: page_1, CTR 0.11 vs 0.16, 201k impressions; trend `stable`. Wrong if this page is a stable high-volume asset that never had strong CTR and isn't actually declining.
5. **#5** `content_91652435f57a` — `refresh_review`: same client, same pattern (page_1, CTR 0.06 vs 0.16); trend `stable`. Wrong for the same reason as #4 — worth checking whether this client's whole page-1 cohort just runs low CTR.
6. **#6** `content_f42eb861c6dd` — `refresh_review`: page_1, position 6.5, CTR 0.13 vs 0.16, label agrees (`down`). Wrong if the CTR gap (3 points) is too small to matter at this position — it's the mildest underperformance in the top 10.
7. **#7** `content_11fcfd65d94c` — `refresh_review`: page_1, CTR 0.15 vs 0.16 — barely underperforming; label agrees (`down`). Wrong for the same reason as #6, and because a 1-point CTR gap could easily be noise week to week.
8. **#8** `content_97a86caf3a3d` — `refresh_review`: page_1, CTR 0.07 vs 0.16, label agrees (`down`). A clean case: clear CTR gap, stale, declining. Wrong if the page has already been refreshed since this snapshot was taken (staleness clock reset) and the score is just stale metrics.
9. **#9** `content_8b36799b7e44` — `refresh_review`: `page_3_5` tier, position 32.0, CTR 0.02 vs 0.03 — the gap is tiny in absolute terms even though it's the same relative story; label agrees (`down`). Wrong if a 1-point CTR gap this far down the results page is just noise, not a real opportunity.
10. **#10** `content_c1fe78bc4e37` — `refresh_review`: page_1, CTR 0.03 vs 0.16 — the widest CTR gap in the top 10, label agrees (`down`). Wrong if the low CTR reflects a mismatched search intent (wrong page ranking for that query) rather than something a content refresh can fix.

## 4. Weak picks + leakage check

In [15]:
# Concentration check: is the top of the queue dominated by a couple of clients?
print("client_id counts in top 20:")
print(ranked.head(20)["client_id"].value_counts())
print()
print("days_since_last_update values in top 20:")
print(ranked.head(20)["days_since_last_update"].value_counts())
print()
print("trend_direction mix in top 20 (label agreement, not used by the rule):")
print(ranked.head(20)["trend_direction"].value_counts())

client_id counts in top 20:
client_id
client_19581e27de    14
client_6208ef0f77     5
client_4e07408562     1
Name: count, dtype: int64

days_since_last_update values in top 20:
days_since_last_update
104    20
Name: count, dtype: int64

trend_direction mix in top 20 (label agreement, not used by the rule):
trend_direction
down      13
stable     4
up         3
Name: count, dtype: int64


In [16]:
import pandas as pd

# Segmented Precision Check: Does the rule perform differently by tier?
tiers = df[df['action_score'] > 0]['position_tier'].unique()
segment_results = []

for tier in tiers:
    tier_mask = df['position_tier'] == tier
    tier_df = df[tier_mask].copy()
    y_tier = tier_df['is_declining_label'].values
    base_tier = y_tier.mean()

    # Precision@20 for this specific tier
    p20 = precision_at_k(tier_df['action_score'], y_tier, 20)

    segment_results.append({
        'tier': tier,
        'n_flagged': (tier_df['action_score'] > 0).sum(),
        'base_decline_rate': base_tier,
        'p@20': p20,
        'lift': p20 - base_tier
    })

segment_df = pd.DataFrame(segment_results).sort_values('lift', ascending=False)
display(segment_df)

,tier,n_flagged,base_decline_rate,p@20,lift
2,striking,529,0.609529,1.00,0.390471
1,page_3_5,528,0.561585,0.90,0.338415
0,page_1,962,0.569663,0.55,-0.019663


### Segment Analysis Results
This check identifies if the rule's effectiveness is concentrated in specific tiers. A high lift in `page_1` confirms that the relative CTR signal is most potent where traffic is highest, while a lower lift in `deep` or `striking` tiers might suggest that the 500-impression threshold or the median CTR comparison is noisier there.

**Weak picks.** Two things jump out that a skeptic should flag:

- **Client concentration.** The top 20 is almost entirely two clients (`client_19581e27de` and `client_6208ef0f77`). Because the score is `stale x visible x ctr_underperform x impressions`, a client with a lot of high-traffic, page-1 content will dominate purely on exposure, even if a smaller client has a page in *more urgent* trouble. A K-per-client cap or a log-scaled impressions term would spread this out.
- **Tied staleness.** Every row in the top 20 has `days_since_last_update == 104`. That's not a rule bug — it's a real batch-update artifact in the data (many pages last touched on the same day) — but it means the ranking within the top 20 is really being decided by impressions alone, not by staleness. The staleness *bar* (≥90 days) is doing its job as a filter; it isn't doing any work as a tiebreaker.
- **4 of the top 20 are labeled `stable` and 3 `up`**, not `down` — those seven are the label mismatches: the rule caught real CTR-vs-position underperformance, but the page's traffic trend hadn't actually turned down yet. That's consistent with the rule being a *leading* indicator on CTR, not a lagging confirmation of decline — worth saying plainly rather than papering over.

**Leakage check.** The score uses `days_since_last_update`, `impressions_90d`, `position_tier`, and `ctr` — all observable *before* the outcome, and none of them derived from `trend_direction` / `trend_pct` / `is_declining_label`. Those three appear in this notebook only in the evaluation cells (signal-check bucket tables, precision@K), never in `action_score`, `reason_code`, or `action_label`. No FlyRank product flags (`health_score`, `needs_ctr_fix`, `is_quick_win`, `priority_score`, `action_type`) exist in the starter dataset in the first place, so there's nothing to accidentally leak from that direction either. No future window is used — every input is a trailing 90-day total, the same window the label's own 30-day comparison sits inside of.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.